# RAG Pipeline Exercise — Build a Question-Answering System

## What is RAG?

Large Language Models (like GPT-4) are powerful, but they have a big limitation: they can only answer from what they memorized during training. If you ask about something they haven't seen, they might **hallucinate** — confidently make up an answer that sounds right but is wrong.

**Retrieval-Augmented Generation (RAG)** solves this by adding a search step. Instead of relying on the LLM's memory, we:

1. **Search** a knowledge base for passages relevant to the user's question
2. **Inject** those passages into the LLM's prompt as context
3. **Ask** the LLM to answer based on what it just read

Think of it like an open-book exam: instead of answering from memory, the LLM gets to read relevant documents first.

## The Pipeline

```
Documents → Chunk → Embed → Store in Vector DB
                                    ↓
User Query → Embed Query → Retrieve Top-k → Augment Prompt → LLM → Answer
```

Here's what each step does:
- **Chunk**: split long documents into shorter passages (~500 characters each)
- **Embed**: convert each passage into a numerical vector that captures its meaning
- **Store**: save all vectors in a database optimized for similarity search
- **Retrieve**: when a user asks a question, find the most similar passages
- **Augment**: insert those passages into the LLM's prompt
- **Answer**: the LLM reads the passages and generates an answer

## How to work through this notebook

Most of the boilerplate code is already written. Look for **`# TODO`** comments — those are the parts you need to fill in. There are also **Discussion** questions after each task: answer them in your own words.

**Estimated time:** 2–3 hours.

---
## Setup

Run the cells below to install dependencies, import libraries, and set your API key.

**Note on imports:**
- `datasets` — HuggingFace library for loading datasets
- `langchain` — framework for building LLM pipelines
- `chromadb` — vector database for storing and searching embeddings
- `helper` — our utility module with plotting and display functions

In [ ]:
# Run this cell once to install all required packages (uncomment the line below)
# !pip install datasets langchain langchain-openai langchain-community chromadb openai tiktoken matplotlib numpy

In [ ]:
import os
import helper
from helper import format_docs

from datasets import load_dataset

# Text splitting
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Embeddings and LLM (both from OpenAI)
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# Vector store
from langchain_community.vectorstores import Chroma

# Chain components
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
# Set your OpenAI API key here.
# You can get one at https://platform.openai.com/api-keys
# os.environ["OPENAI_API_KEY"] = "sk-..."

assert os.environ.get("OPENAI_API_KEY"), "Set your OPENAI_API_KEY before continuing."

---
## Task 1 — Data Loading & Exploration

### What we're doing

We need a **knowledge base** — a collection of documents that our RAG pipeline will search through. We'll use Wikipedia paragraphs from the SQuAD dataset.

SQuAD contains thousands of question–answer pairs, where each question refers to a Wikipedia paragraph. For example:
- **Paragraph**: "The Normans were the people who gave their name to Normandy, a region in France..."
- **Question**: "In what country is Normandy located?"
- **Answer**: "France"

Multiple questions can refer to the same paragraph, so we extract and **deduplicate** the paragraphs to get our unique documents.

### Nothing to code here

Just run the cells below and observe the output. Pay attention to:
- How many unique documents we have
- What a typical document looks like
- How document lengths are distributed

In [ ]:
# Load the SQuAD v2 validation split from HuggingFace
# This downloads the dataset the first time you run it
squad = load_dataset("rajpurkar/squad_v2", split="validation")
print(f"Total examples in validation set: {len(squad)}")

In [ ]:
# Extract unique documents (Wikipedia paragraphs).
# dict.fromkeys() preserves order and removes duplicates.
all_documents = list(dict.fromkeys(squad["context"]))
print(f"Unique documents: {len(all_documents)}")

# We take a subset of 200 documents to keep things fast.
# (Embedding all of them would be slow and expensive.)
documents = all_documents[:200]
print(f"Working set: {len(documents)} documents")
print(f"\nHere's the first document (first 300 chars):")
print(f"{documents[0][:300]}...")

In [ ]:
# Visualize document length distribution.
# This shows a histogram + summary statistics.
helper.plot_document_length_distribution(documents)

### Discussion

Look at the histogram above.
- What is the typical length of a document (in characters)?
- Are the documents all roughly the same length, or do they vary a lot?
- Why might this matter for the next step (chunking)?

*Your answer:* [write here]

---
## Task 2 — Document Chunking

### Why we chunk

We can't just embed entire documents as-is for two reasons:

1. **Embedding models have a maximum input length.** If a document is too long, the model can't process it.
2. **Retrieval is more precise with shorter chunks.** A 500-character chunk about "the French Revolution" will match a question about that topic better than a 5000-character document that mentions it briefly alongside many other topics.

### How `RecursiveCharacterTextSplitter` works

LangChain's `RecursiveCharacterTextSplitter` is a smart splitter: instead of cutting text at arbitrary positions, it tries to split at natural boundaries in this order:
1. Double newlines (paragraph breaks)
2. Single newlines
3. Spaces (between words)
4. Individual characters (last resort)

This keeps chunks coherent and readable.

**Parameters:**
- `chunk_size=500` — target maximum length of each chunk (in characters)
- `chunk_overlap=50` — how many characters the end of one chunk and the start of the next one share. This prevents important information from being lost at chunk boundaries.

### Your task

Fill in the two `# TODO` lines below:
1. Create the splitter with the parameters described above.
2. Use it to split the documents into chunks.

In [ ]:
# TODO: Create a RecursiveCharacterTextSplitter with chunk_size=500 and chunk_overlap=50
#
# Example:
#   text_splitter = RecursiveCharacterTextSplitter(
#       chunk_size=...,
#       chunk_overlap=...,
#   )
text_splitter = # TODO

# TODO: Split the documents into chunks.
# The method is: text_splitter.create_documents(documents)
# It returns a list of LangChain Document objects.
chunks = # TODO

print(f"We split {len(documents)} documents into {len(chunks)} chunks.")
print(f"\nHere's what the first chunk looks like:")
print(f"{chunks[0].page_content[:200]}...")

In [ ]:
# Visualize chunk lengths — are they close to our target of 500 characters?
helper.plot_chunk_length_distribution(chunks)

### Discussion

Think about the trade-offs in chunk size:
- What would happen if chunks were **very small** (e.g., 50 characters — just a sentence fragment)? Would retrieval work well?
- What would happen if chunks were **very large** (e.g., 5000 characters — entire articles)? Would the LLM be able to find the relevant information?
- Why did we choose `chunk_overlap=50`? What problem does it solve?

*Your answer:* [write here]

---
## Task 3 — Embeddings & Vector Store

### What are embeddings?

An **embedding** is a way to represent text as a list of numbers (a vector). The key property is that **texts with similar meanings get similar vectors**.

For example, the sentences "The cat sat on the mat" and "A kitten rested on the rug" would have very similar embedding vectors, even though they use completely different words. Meanwhile, "The stock market crashed" would have a very different vector.

This is what makes retrieval possible: we embed both the chunks and the user's question, then find the chunks whose vectors are closest to the question's vector.

### What is a vector store?

A **vector store** (like ChromaDB) is a database designed for storing and searching vectors efficiently. When you call `similarity_search`, it finds the vectors in the database that are most similar to your query vector.

### What happens when you run `Chroma.from_documents()`

1. Each chunk's text is sent to the embedding model (OpenAI's API)
2. The model returns a vector for each chunk
3. ChromaDB stores both the text and the vector
4. Later, when you search, ChromaDB converts your query to a vector and finds the closest stored vectors

### Your task

Fill in the two `# TODO` lines to create the embedding model and build the vector store.

In [ ]:
# TODO: Create an embedding model.
# We use OpenAI's "text-embedding-3-small" model — it's fast and cheap.
#
# Example:
#   embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings = # TODO

# TODO: Build the vector store from your chunks.
# This will send all chunks to the embedding model and store the results.
# It might take a minute depending on how many chunks you have.
#
# Example:
#   vectorstore = Chroma.from_documents(chunks, embeddings)
vectorstore = # TODO

print(f"Vector store ready — {vectorstore._collection.count()} vectors indexed.")

### Testing retrieval

Let's test the vector store by searching for some queries. We use `similarity_search_with_score` which returns both the matching documents and their distance scores.

**How to read the scores:** ChromaDB uses **distance** (not similarity), so **lower scores = more similar**. A score close to 0 means the query and document are very similar; a high score means they're not related.

We test with three queries:
- Two **in-domain** queries (topics that should be in our Wikipedia knowledge base)
- One **out-of-domain** query (pizza toppings — definitely not in Wikipedia's SQuAD subset)

In [ ]:
# Test retrieval with different types of queries
test_queries = [
    "Who was the first president of the United States?",
    "What is photosynthesis?",
    "What is the best pizza topping?",  # out-of-domain!
]

for query in test_queries:
    results = vectorstore.similarity_search_with_score(query, k=3)
    helper.display_retrieval_results(query, results)

### Discussion

Look at the retrieval results above:
- For the in-domain queries, do the retrieved passages seem relevant? Do they contain information that could help answer the question?
- For the out-of-domain query (pizza), what happens? The vector store always returns *something* — does it look useful?
- Look at the scores: are they different between in-domain and out-of-domain queries?

*Your answer:* [write here]

---
## Task 4 — Build the RAG Chain

### How the chain works

Now we connect all the pieces into a working pipeline using LangChain's **LCEL** (LangChain Expression Language). LCEL lets you chain components together using the pipe `|` operator, similar to Unix pipes.

Here's what happens when you call `rag_chain.invoke("some question")`:

```
Step 1: The retriever searches the vector store for chunks relevant to the question
Step 2: format_docs joins those chunks into a single "context" string
Step 3: The prompt template inserts the context and question into a structured prompt
Step 4: The LLM reads the prompt and generates an answer
Step 5: The output parser extracts just the text from the LLM's response
```

### The retriever

A **retriever** wraps the vector store so it can be used in a chain. When called, it takes a question string and returns a list of relevant Document objects. The `k=4` parameter means it returns the 4 most relevant chunks.

### Your task

Write the **prompt template**. This is the most important creative decision in RAG — the prompt tells the LLM how to behave. A good prompt template should:

1. Include `{context}` and `{question}` placeholders (LangChain will fill these in automatically)
2. Tell the LLM to answer **only** based on the provided context
3. Tell the LLM to say "I don't know" if the context doesn't contain the answer

**Why rule 2 matters:** without it, the LLM might ignore the context and answer from its own memory, defeating the purpose of RAG.

**Why rule 3 matters:** without it, the LLM might hallucinate an answer when the retrieved passages aren't relevant (like our pizza question).

In [ ]:
# Create a retriever from the vector store.
# k=4 means it will retrieve the 4 most relevant chunks for each question.
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Initialize the LLM.
# We use gpt-4o-mini because it's fast, cheap, and good enough for this exercise.
# temperature=0 makes the output deterministic (always picks the most likely answer).
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
# TODO: Write your prompt template.
#
# Replace the "TODO: write your prompt here" text with actual instructions.
# The {context} and {question} placeholders must stay — LangChain fills them in.
#
# A good prompt might look something like:
#   "You are a helpful assistant. Use the context below to answer the question.
#    If the context doesn't contain the answer, say 'I don't know'.
#    Context: {context}
#    Question: {question}"
#
# Feel free to write it in your own style!

RAG_PROMPT_TEMPLATE = """
TODO: write your prompt here.

Context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)

In [ ]:
# Build the RAG chain using LCEL.
# This is already done for you — don't modify it.
#
# How to read this:
#   1. {"context": ..., "question": ...} creates a dict with two keys
#   2. retriever | format_docs: retrieves chunks, then joins them into a string
#   3. RunnablePassthrough(): passes the question through unchanged
#   4. | prompt: fills in the template with context and question
#   5. | llm: sends the filled prompt to the LLM
#   6. | StrOutputParser(): extracts just the text from the LLM response

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# Test the chain with some questions!
test_questions = [
    "Who was the first president of the United States?",
    "What is photosynthesis?",
    "What is the best pizza topping?",  # out-of-domain
]

for q in test_questions:
    answer = rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

### Discussion

Look at the answers:
- Did the chain answer the in-domain questions correctly?
- Did it say "I don't know" for the out-of-domain question (pizza)? If not, how could you improve your prompt?
- Try modifying your prompt template above and re-running the cells. Does the behavior change?

*Your answer:* [write here]

---
## Task 5 — Evaluation & Comparison

### What we're measuring

Testing with 3 questions is not enough to know if our pipeline works well. We need a systematic evaluation.

We use **substring accuracy**: for each question, we check if the expected answer appears anywhere in the generated answer (case-insensitive). For example:
- Gold answer: `"France"`
- Generated: `"The answer is France, a country in Europe"` → **correct** ("france" is in the text)
- Generated: `"Germany"` → **incorrect** ("france" is not in the text)

This is a simple but effective metric. It's not perfect (the LLM might say "France" in a wrong context), but it gives us a reasonable estimate.

### RAG vs. Naive comparison

To understand the value of retrieval, we compare our RAG chain against a **naive chain** — one that sends the question directly to the LLM without any context. This tells us: does retrieval actually help, or could the LLM answer just as well from memory?

### Your tasks

1. **Build a naive chain** (the `# TODO` cell): create a simple prompt with just `{question}`, no `{context}`. Pipe it through the same LLM and output parser.
2. **Compute substring accuracy** (the `# TODO` cell): count how many gold answers appear in the generated answers.

In [ ]:
# Build an evaluation set.
# This finds SQuAD questions whose context paragraph is in our knowledge base,
# so we know the answer SHOULD be retrievable.
eval_set = helper.build_eval_set(squad, documents, n=30)
print(f"\nEvaluation set: {len(eval_set)} questions")
print(f"\nExample:")
print(f"  Question: {eval_set[0]['question']}")
print(f"  Answer:   {eval_set[0]['answer']}")

In [ ]:
# Run the RAG chain on 10 evaluation questions.
# We use 10 to keep API costs low while still getting meaningful results.
eval_questions = [ex["question"] for ex in eval_set[:10]]
gold_answers = [ex["answer"] for ex in eval_set[:10]]

print("Running RAG chain on evaluation questions...\n")
rag_answers = []
for i, q in enumerate(eval_questions):
    answer = rag_chain.invoke(q)
    rag_answers.append(answer)
    print(f"[{i+1}/10] Q: {q}")
    print(f"       A: {answer}")
    print(f"    Gold: {gold_answers[i]}\n")

In [ ]:
# TODO: Build a naive chain — sends only the question to the LLM, without any context.
#
# Step 1: Create a simple prompt template with just {question}.
#         Example: ChatPromptTemplate.from_template("Answer this question concisely:\n\n{question}")
#
# Step 2: Chain it: naive_prompt | llm | StrOutputParser()

naive_prompt = # TODO
naive_chain = # TODO

In [ ]:
# Run the naive chain on the same questions
print("Running naive chain (no retrieval) on the same questions...\n")
naive_answers = []
for i, q in enumerate(eval_questions):
    answer = naive_chain.invoke(q)
    naive_answers.append(answer)
    print(f"[{i+1}/10] Q: {q}")
    print(f"       A: {answer}\n")

In [ ]:
# Display the comparison table
# ✓ means the gold answer was found in the generated answer
# ✗ means it was not
helper.display_comparison_table(eval_questions, rag_answers, naive_answers, gold_answers)

In [ ]:
# TODO: Compute substring accuracy for both chains.
#
# "Substring accuracy" = fraction of questions where gold_answer.lower()
# appears inside generated_answer.lower().
#
# Hint: you can use a list comprehension or a loop. For example:
#   rag_accuracy = sum(
#       1 for ra, ga in zip(rag_answers, gold_answers)
#       if ga.lower() in ra.lower()
#   ) / len(eval_questions)

rag_accuracy = # TODO
naive_accuracy = # TODO

print(f"RAG accuracy:   {rag_accuracy:.1%}")
print(f"Naive accuracy: {naive_accuracy:.1%}")

In [ ]:
# Visualize the comparison
helper.plot_evaluation_results({
    "RAG Accuracy": rag_accuracy,
    "Naive Accuracy": naive_accuracy,
})

### Discussion

Compare the RAG and naive results:
- Which chain performed better overall? By how much?
- Look at the comparison table: are there questions where the naive chain got it right but RAG didn't (or vice versa)? Why might that happen?
- What kinds of questions benefit most from retrieval? (Hint: think about whether the answer is "common knowledge" vs. specific factual details.)
- Can you think of a real-world application where RAG would be essential?

*Your answer:* [write here]